In [ ]:
# Requirements: python>=3.8, torch, numpy, scipy
import os, sys, json, time, math, random, hashlib, resource, platform
import warnings
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats as sps

warnings.filterwarnings("ignore")
SEEDS = [42, 123, 456]
CACHE_DIR = "/kaggle/working/dqn_society_cache" if os.path.isdir("/kaggle/working") else "./dqn_society_cache"
CODE_VERSION = "v1"
ACTIONS = ("collect", "invest", "reproduce")

BASE = dict(
    system="free", groups=0, split="edu6040", policy="dqn",
    n_agents=50, days=365, iterations=3, food_lo=1000, food_hi=3000,
    max_age_lo=365, max_age_hi=3650, wealth_lo=10, wealth_hi=50, edu_lo=0.0, edu_hi=30.0,
    needs_lo=10, needs_hi=40, days_until_death=2, invest_base=0.5, invest_gain=0.1,
    repro_wealth=200, cooldown=365, collect_eff=0.08, edu_gain_collect=0.01, edu_gain_pool=0.005,
    partner_prob=0.15, lr=1e-3, gamma=0.99, eps_start=0.9, eps_end=0.05, eps_decay_days=730,
    eps_clock="iteration", cooldown_clock="iteration", buffer=1000, batch=32, hidden=64,
    target_update=100, grad_clip=100.0, r_invest_success=0.2, r_invest_fail=-0.1, r_repro=0.5,
    happiness_term=True, clip_state=False, pop_cap=3000)

DEVICE = torch.device("cpu")


def out(line, sink=None):
    print(line, flush=True)
    if sink is not None:
        sink.append(line)


def cfg_of(**kw):
    c = dict(BASE)
    c.update(kw)
    return c


def run_key(exp, label, cfg, seed):
    blob = json.dumps(dict(cfg=cfg, seed=seed, v=CODE_VERSION), sort_keys=True)
    return f"{exp}__{label}__s{seed}__{hashlib.md5(blob.encode()).hexdigest()[:10]}"


def gini(values):
    v = np.sort(np.asarray(values, dtype=np.float64))
    n = len(v)
    if n == 0 or v.sum() <= 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * v)) / (n * v.sum()) - (n + 1) / n)


class QNet(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc1 = nn.Linear(5, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 3)

    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))


class Agent:
    __slots__ = ("id", "group_id", "age", "max_age", "wealth", "education", "needs", "children", "happiness",
                 "days_without_food", "alive", "last_repro", "cohort0", "net", "target", "opt", "mem", "steps", "c")

    def __init__(self, aid, c, group_id=0, cohort0=False):
        self.id, self.group_id, self.c, self.cohort0 = aid, group_id, c, cohort0
        self.age = random.randint(0, 100)
        self.max_age = random.randint(c["max_age_lo"], c["max_age_hi"])
        self.wealth = random.randint(c["wealth_lo"], c["wealth_hi"])
        self.education = random.uniform(c["edu_lo"], c["edu_hi"])
        self.needs = random.randint(c["needs_lo"], c["needs_hi"])
        self.children, self.happiness, self.days_without_food, self.alive = 0, 0.0, 0, True
        self.last_repro = -c["cooldown"]
        self.steps = 0
        if c["policy"] == "dqn":
            self.net = QNet(c["hidden"]).to(DEVICE)
            self.target = QNet(c["hidden"]).to(DEVICE)
            self.target.load_state_dict(self.net.state_dict())
            self.opt = torch.optim.Adam(self.net.parameters(), lr=c["lr"])
            self.mem = deque(maxlen=c["buffer"])
        else:
            self.net = self.target = self.opt = self.mem = None
        self.calc_happiness()

    def calc_happiness(self):
        self.happiness = (min(self.wealth / 1000.0, 1.0) * 0.4 + (self.education / 100.0) * 0.4
                          + min(self.children / 5.0, 1.0) * 0.2)
        return self.happiness

    def state(self, food):
        s = [self.wealth / 10000.0, self.age / float(self.c["max_age_hi"]), self.education / 100.0,
             self.needs / 100.0, food / 3000.0]
        if self.c["clip_state"]:
            s = [min(max(x, 0.0), 1.0) for x in s]
        return s

    def can_reproduce(self, clock_day):
        return self.wealth >= self.c["repro_wealth"] and clock_day - self.last_repro >= self.c["cooldown"]

    def optimize(self):
        c = self.c
        if len(self.mem) < c["batch"]:
            return None
        batch = random.sample(self.mem, c["batch"])
        s = torch.tensor([b[0] for b in batch], dtype=torch.float32, device=DEVICE)
        a = torch.tensor([[b[1]] for b in batch], dtype=torch.long, device=DEVICE)
        r = torch.tensor([[b[2]] for b in batch], dtype=torch.float32, device=DEVICE)
        s2 = torch.tensor([b[3] for b in batch], dtype=torch.float32, device=DEVICE)
        nd = torch.tensor([[0.0 if b[4] else 1.0] for b in batch], dtype=torch.float32, device=DEVICE)
        q = self.net(s).gather(1, a)
        with torch.no_grad():
            tgt = r + c["gamma"] * self.target(s2).max(1, keepdim=True)[0] * nd
        loss = F.mse_loss(q, tgt)
        self.opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_value_(self.net.parameters(), c["grad_clip"])
        self.opt.step()
        self.steps += 1
        if self.steps % c["target_update"] == 0:
            self.target.load_state_dict(self.net.state_dict())
        return float(loss.item())


class Society:
    def __init__(self, c):
        self.c = c
        self.pooled = c["system"] in ("centralized", "hybrid")
        self.n_groups = 1 if c["system"] == "centralized" else (c["groups"] if c["system"] == "hybrid" else 0)
        self.agents = [Agent(i, c, cohort0=True) for i in range(c["n_agents"])]
        self.groups = [[] for _ in range(self.n_groups)]
        if self.pooled:
            for i, ag in enumerate(self.agents):
                ag.group_id = i % self.n_groups
                self.groups[ag.group_id].append(ag)
        self.next_id = c["n_agents"]
        self.total_births = 0
        self.total_deaths = dict(starvation=0, age=0)
        self.cap_hit = False
        self.reset_flow()

    def reset_flow(self):
        self.flow = dict(births=0, d_starv=0, d_age=0, loss_sum=0.0, loss_n=0, q_sum=0.0, q_n=0, r_sum=0.0, r_n=0,
                         act=[0, 0, 0], greedy=[0, 0, 0])

    def epsilon(self, day, gday):
        c = self.c
        d = day if c["eps_clock"] == "iteration" else gday
        if d < c["eps_decay_days"]:
            return c["eps_start"] - (c["eps_start"] - c["eps_end"]) * (d / c["eps_decay_days"])
        return c["eps_end"]

    def choose(self, ag, s, eps, clock_day):
        pol = self.c["policy"]
        if pol == "dqn":
            with torch.no_grad():
                q = ag.net(torch.tensor([s], dtype=torch.float32, device=DEVICE))[0]
            g = int(q.argmax().item())
            self.flow["q_sum"] += float(q.max().item())
            self.flow["q_n"] += 1
            self.flow["greedy"][g] += 1
            if random.random() > eps:
                return g
            return random.randint(0, 2)
        if pol == "random":
            return random.randint(0, 2)
        if ag.can_reproduce(clock_day):
            return 2
        if ag.wealth < self.c["repro_wealth"]:
            return 0
        return 1

    def step_day(self, day, gday):
        c, f = self.c, self.flow
        food = random.randint(c["food_lo"], c["food_hi"])
        clock_day = day if c["cooldown_clock"] == "iteration" else gday
        if self.pooled:
            for group in self.groups:
                alive = [a for a in group if a.alive]
                if not alive:
                    continue
                coll = []
                for ag in alive:
                    coll.append((ag, random.uniform(0, food * c["collect_eff"])))
                    ag.education += c["edu_gain_pool"]
                total = sum(x for _, x in coll)
                needs = sum(a.needs for a in alive)
                if total >= needs:
                    surplus = total - needs
                    if c["split"] == "edu6040":
                        ranked = sorted(alive, key=lambda a: a.education, reverse=True)
                        top = max(1, len(ranked) // 5)
                        for i, ag in enumerate(ranked):
                            if i < top:
                                ag.wealth += surplus * 0.6 / top
                            else:
                                ag.wealth += surplus * 0.4 / (len(ranked) - top)
                    else:
                        for ag in alive:
                            ag.wealth += surplus / len(alive)
                    for ag in alive:
                        ag.days_without_food = 0
                else:
                    ratio = total / needs if needs > 0 else 0
                    for ag, fc in coll:
                        if fc >= ag.needs * ratio:
                            ag.days_without_food = 0
                        else:
                            ag.days_without_food += 1
                            if ag.days_without_food >= c["days_until_death"]:
                                ag.alive = False
                                f["d_starv"] += 1
        eps = self.epsilon(day, gday)
        newborn = []
        for ag in self.agents:
            if not ag.alive:
                continue
            s = ag.state(food)
            act = self.choose(ag, s, eps, clock_day)
            f["act"][act] += 1
            reward = 0.0
            if act == 0:
                if not self.pooled:
                    got = random.uniform(0, food * c["collect_eff"])
                    if got >= ag.needs:
                        ag.wealth += got - ag.needs
                        ag.education += c["edu_gain_collect"]
                        ag.days_without_food = 0
                    else:
                        ag.days_without_food += 1
                        if ag.days_without_food >= c["days_until_death"]:
                            ag.alive = False
                            f["d_starv"] += 1
                            continue
            elif act == 1:
                amount = min(ag.wealth * 0.1, 100)
                p = min(1.0, c["invest_base"] + ag.education / 100.0)
                if random.random() < p:
                    ag.wealth += amount * c["invest_gain"]
                    reward = c["r_invest_success"]
                else:
                    ag.wealth -= amount
                    reward = c["r_invest_fail"]
            else:
                if ag.can_reproduce(clock_day):
                    pool = self.groups[ag.group_id] if self.pooled else self.agents
                    partners = [a for a in pool if a is not ag and a.alive and a.can_reproduce(clock_day)
                                and random.random() < c["partner_prob"]]
                    if partners and (len(self.agents) + len(newborn)) < c["pop_cap"]:
                        partner = random.choice(partners)
                        child_edu = max(0, min((ag.education + partner.education) / 2 + random.uniform(-5, 5), 100))
                        child_wealth = max(0, (ag.wealth + partner.wealth) / 4)
                        ag.children += 1
                        partner.children += 1
                        ag.last_repro = clock_day
                        partner.last_repro = clock_day
                        child = Agent(self.next_id, c, group_id=ag.group_id)
                        self.next_id += 1
                        child.education, child.wealth = child_edu, child_wealth
                        newborn.append(child)
                        f["births"] += 1
                        reward = c["r_repro"]
                    elif partners:
                        self.cap_hit = True
            old_h = ag.happiness
            ag.age += 1
            if ag.age >= ag.max_age:
                ag.alive = False
                f["d_age"] += 1
                s2, done = [0.0] * 5, True
            else:
                ag.calc_happiness()
                s2, done = ag.state(food), False
            if c["happiness_term"]:
                reward += ag.happiness - old_h
            f["r_sum"] += reward
            f["r_n"] += 1
            if c["policy"] == "dqn":
                ag.mem.append((s, act, reward, s2, done))
                loss = ag.optimize()
                if loss is not None:
                    f["loss_sum"] += loss
                    f["loss_n"] += 1
        self.agents = [a for a in self.agents if a.alive]
        for ch in newborn:
            self.agents.append(ch)
            if self.pooled:
                self.groups[ch.group_id].append(ch)
        if self.pooled:
            self.groups = [[a for a in g if a.alive] for g in self.groups]

    def snapshot(self):
        w = [a.wealth for a in self.agents]
        h = [a.happiness for a in self.agents]
        e = [a.education for a in self.agents]
        n = len(w)
        return dict(pop=n, wealth=float(np.mean(w)) if n else 0.0, happy=float(np.mean(h)) if n else 0.0,
                    edu=float(np.mean(e)) if n else 0.0, gini=gini(w) if n else 0.0)

    def iteration_summary(self, itflow):
        c = self.c
        w = np.array([a.wealth for a in self.agents], dtype=np.float64)
        h = np.array([a.happiness for a in self.agents], dtype=np.float64)
        e = np.array([a.education for a in self.agents], dtype=np.float64)
        n = len(w)
        ever = c["n_agents"] + self.total_births
        dead = self.total_deaths["starvation"] + self.total_deaths["age"]
        cohort_alive = sum(1 for a in self.agents if a.cohort0)
        r = dict(pop=n, births_iter=itflow["births"], deaths_starv_iter=itflow["d_starv"], deaths_age_iter=itflow["d_age"],
                 births_cum=self.total_births, deaths_cum=dead, deaths_starv_cum=self.total_deaths["starvation"],
                 deaths_age_cum=self.total_deaths["age"], net_change=n - c["n_agents"], ever_lived=ever,
                 mortality_ever_lived=dead / ever if ever else 0.0, cohort0_survivors=cohort_alive,
                 cohort0_mortality=1 - cohort_alive / c["n_agents"])
        if n:
            ws = np.sort(w)
            top = max(1, n // 5)
            order = np.argsort(e)
            lo_half, hi_half = w[order[: n // 2]], w[order[n // 2:]]
            r.update(wealth_mean=float(w.mean()), wealth_median=float(np.median(w)), wealth_std=float(w.std()),
                     wealth_min=float(w.min()), wealth_max=float(w.max()), gini=gini(w),
                     gini_ever_lived=gini(np.concatenate([w, np.zeros(ever - n)])),
                     max_min_ratio=float(w.max() / w.min()) if w.min() > 0 else float("nan"),
                     top20_share=float(ws[-top:].sum() / ws.sum()) if ws.sum() > 0 else float("nan"),
                     edu_half_wealth_ratio=float(hi_half.mean() / lo_half.mean()) if len(lo_half) and lo_half.mean() > 0 else float("nan"),
                     happy_mean=float(h.mean()), happy_std=float(h.std()), happy_min=float(h.min()), happy_max=float(h.max()),
                     happy_mean_ever_lived=float(h.sum() / ever), edu_mean=float(e.mean()), edu_std=float(e.std()))
        else:
            for k in ("wealth_mean", "wealth_median", "wealth_std", "wealth_min", "wealth_max", "gini", "max_min_ratio",
                      "top20_share", "edu_half_wealth_ratio", "happy_mean", "happy_std", "happy_min", "happy_max",
                      "happy_mean_ever_lived", "edu_mean", "edu_std"):
                r[k] = 0.0
            r["gini_ever_lived"] = 0.0
        ta, tg = sum(itflow["act"]), sum(itflow["greedy"])
        for i, nm in enumerate(ACTIONS):
            r[f"act_{nm}"] = itflow["act"][i] / ta if ta else 0.0
            r[f"greedy_{nm}"] = itflow["greedy"][i] / tg if tg else float("nan")
        r["loss_mean"] = itflow["loss_sum"] / itflow["loss_n"] if itflow["loss_n"] else float("nan")
        r["maxq_mean"] = itflow["q_sum"] / itflow["q_n"] if itflow["q_n"] else float("nan")
        r["reward_mean"] = itflow["r_sum"] / itflow["r_n"] if itflow["r_n"] else float("nan")
        r["agent_steps"] = itflow["r_n"]
        return r, w, h


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def fmt(x, nd=4):
    if isinstance(x, float):
        return "nan" if math.isnan(x) else f"{x:.{nd}f}"
    return str(x)


def simulate(exp, label, cfg, seed, sink, dump_dist):
    tag = f"[{exp}|{label}|seed={seed}]"
    seed_all(seed)
    t0 = time.time()
    soc = Society(cfg)
    out(f"{tag} RUN_START system={cfg['system']} groups={soc.n_groups} split={cfg['split']} policy={cfg['policy']} "
        f"n_agents={cfg['n_agents']} days={cfg['days']} iterations={cfg['iterations']}", sink)
    summaries = []
    gday = 0
    for it in range(1, cfg["iterations"] + 1):
        itflow = dict(births=0, d_starv=0, d_age=0, loss_sum=0.0, loss_n=0, q_sum=0.0, q_n=0, r_sum=0.0, r_n=0,
                      act=[0, 0, 0], greedy=[0, 0, 0])
        soc.reset_flow()
        for day in range(cfg["days"]):
            soc.step_day(day, gday)
            gday += 1
            if day % 30 == 0 or day == cfg["days"] - 1:
                f, sn = soc.flow, soc.snapshot()
                ta, tg = sum(f["act"]), sum(f["greedy"])
                out(f"{tag} it={it} day={day:03d} month={day // 30:02d} pop={sn['pop']} births={f['births']} "
                    f"deaths_starv={f['d_starv']} deaths_age={f['d_age']} wealth={sn['wealth']:.2f} happy={sn['happy']:.4f} "
                    f"edu={sn['edu']:.3f} gini={sn['gini']:.4f} eps={soc.epsilon(day, gday - 1):.4f} "
                    f"loss={fmt(f['loss_sum'] / f['loss_n'] if f['loss_n'] else float('nan'), 6)} "
                    f"maxQ={fmt(f['q_sum'] / f['q_n'] if f['q_n'] else float('nan'), 5)} "
                    f"reward={fmt(f['r_sum'] / f['r_n'] if f['r_n'] else float('nan'), 6)} "
                    f"act_c/i/r={'/'.join(fmt(x / ta if ta else 0.0, 3) for x in f['act'])} "
                    f"greedy_c/i/r={'/'.join(fmt(x / tg if tg else float('nan'), 3) for x in f['greedy'])}", sink)
                for k in ("births", "d_starv", "d_age", "loss_sum", "loss_n", "q_sum", "q_n", "r_sum", "r_n"):
                    itflow[k] += f[k]
                for i in range(3):
                    itflow["act"][i] += f["act"][i]
                    itflow["greedy"][i] += f["greedy"][i]
                soc.total_births += f["births"]
                soc.total_deaths["starvation"] += f["d_starv"]
                soc.total_deaths["age"] += f["d_age"]
                soc.reset_flow()
        summ, w, h = soc.iteration_summary(itflow)
        summaries.append(summ)
        out(f"{tag} ITERATION_SUMMARY it={it} " + " ".join(f"{k}={fmt(v)}" for k, v in summ.items()), sink)
        if dump_dist:
            out(f"{tag} WEALTH_VALUES it={it} n={len(w)} " + ",".join(f"{x:.1f}" for x in np.sort(w)), sink)
            out(f"{tag} HAPPINESS_VALUES it={it} n={len(h)} " + ",".join(f"{x:.4f}" for x in np.sort(h)), sink)
    wall = time.time() - t0
    steps = sum(s["agent_steps"] for s in summaries)
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
    out(f"{tag} RUN_END wall_time_s={wall:.2f} agent_steps={steps} steps_per_s={steps / max(wall, 1e-9):.1f} "
        f"peak_rss_mb={rss:.1f} pop_cap_reached={soc.cap_hit}", sink)
    return dict(summaries=summaries, wall=wall, steps=steps, rss=rss, cap_hit=soc.cap_hit)


def run_cached(exp, label, cfg, seed, dump_dist=False, use_cache=True):
    key = run_key(exp, label, cfg, seed)
    path = os.path.join(CACHE_DIR, key + ".json")
    if use_cache and os.path.exists(path):
        try:
            with open(path) as fh:
                rec = json.load(fh)
            for line in rec["log"]:
                print(line, flush=True)
            print(f"[{exp}|{label}|seed={seed}] RESTORED_FROM_CACHE key={key}", flush=True)
            return rec["result"]
        except Exception as ex:
            print(f"[{exp}|{label}|seed={seed}] CACHE_READ_FAILED key={key} error={ex!r} action=rerun", flush=True)
    sink = []
    res = simulate(exp, label, cfg, seed, sink, dump_dist)
    if use_cache:
        os.makedirs(CACHE_DIR, exist_ok=True)
        tmp = path + ".tmp"
        with open(tmp, "w") as fh:
            json.dump(dict(log=sink, result=res), fh)
        os.replace(tmp, path)
    return res


def ci95(vals):
    v = np.asarray(vals, dtype=np.float64)
    n = len(v)
    if n < 2:
        return float("nan")
    return float(sps.t.ppf(0.975, n - 1) * v.std(ddof=1) / math.sqrt(n))


def hedges_g(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    sp = math.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    if sp < 1e-12:
        return float("nan") if a.mean() == b.mean() else float("inf") * np.sign(a.mean() - b.mean())
    return float((a.mean() - b.mean()) / sp * (1 - 3 / (4 * (na + nb) - 9)))


def compare(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if np.allclose(a, a[0]) and np.allclose(b, b[0]):
        t, p = float("nan"), (1.0 if a[0] == b[0] else float("nan"))
    else:
        t, p = sps.ttest_ind(a, b, equal_var=False)
    try:
        u, pu = sps.mannwhitneyu(a, b, alternative="two-sided")
    except ValueError:
        u, pu = float("nan"), float("nan")
    return float(t), float(p), hedges_g(a, b), float(u), float(pu)


KEY_METRICS = ["pop", "net_change", "births_cum", "deaths_cum", "deaths_starv_cum", "deaths_age_cum", "mortality_ever_lived",
               "cohort0_survivors", "cohort0_mortality", "wealth_mean", "wealth_median", "wealth_std", "wealth_min",
               "wealth_max", "gini", "gini_ever_lived", "max_min_ratio", "top20_share", "edu_half_wealth_ratio",
               "happy_mean", "happy_std", "happy_mean_ever_lived", "edu_mean", "act_collect", "act_invest", "act_reproduce",
               "greedy_collect", "greedy_invest", "greedy_reproduce", "loss_mean", "maxq_mean", "reward_mean"]
TEST_METRICS = ["pop", "gini", "gini_ever_lived", "wealth_mean", "happy_mean", "mortality_ever_lived", "cohort0_mortality",
                "deaths_cum", "births_cum"]


def aggregate(exp, label, results):
    n_it = len(results[0]["summaries"])
    for it in range(n_it):
        for m in KEY_METRICS:
            vals = [float(r["summaries"][it][m]) for r in results]
            v = np.asarray(vals)
            print(f"[{exp}|{label}] AGG it={it + 1} metric={m} values={','.join(fmt(x) for x in vals)} "
                  f"mean={fmt(float(np.nanmean(v)))} sd={fmt(float(np.nanstd(v, ddof=1)) if len(v) > 1 else float('nan'))} "
                  f"ci95_halfwidth={fmt(ci95(vals))}", flush=True)
    walls = [r["wall"] for r in results]
    print(f"[{exp}|{label}] COST wall_time_s_values={','.join(f'{x:.1f}' for x in walls)} mean={np.mean(walls):.1f} "
          f"agent_steps_mean={np.mean([r['steps'] for r in results]):.0f} peak_rss_mb_max={max(r['rss'] for r in results):.1f}",
          flush=True)


def final_vals(results, metric):
    return [float(r["summaries"][-1][metric]) for r in results]


def pairwise(exp, name_a, res_a, name_b, res_b):
    for m in TEST_METRICS:
        a, b = final_vals(res_a, m), final_vals(res_b, m)
        t, p, g, u, pu = compare(a, b)
        print(f"[{exp}] TEST final_iteration metric={m} A={name_a} B={name_b} mean_A={fmt(float(np.mean(a)))} "
              f"mean_B={fmt(float(np.mean(b)))} diff={fmt(float(np.mean(a) - np.mean(b)))} welch_t={fmt(t)} p={fmt(p)} "
              f"hedges_g={fmt(g)} mannwhitney_U={fmt(u)} p_U={fmt(pu)}", flush=True)


def omnibus(exp, named):
    for m in TEST_METRICS:
        groups = [final_vals(r, m) for _, r in named]
        try:
            fstat, pf = sps.f_oneway(*groups)
        except Exception:
            fstat, pf = float("nan"), float("nan")
        try:
            hstat, ph = sps.kruskal(*groups)
        except Exception:
            hstat, ph = float("nan"), float("nan")
        grand = np.mean(np.concatenate([np.asarray(g) for g in groups]))
        ssb = sum(len(g) * (np.mean(g) - grand) ** 2 for g in groups)
        sst = sum(((np.asarray(g) - grand) ** 2).sum() for g in groups)
        print(f"[{exp}] OMNIBUS final_iteration metric={m} groups={'|'.join(n for n, _ in named)} anova_F={fmt(float(fstat))} "
              f"p={fmt(float(pf))} eta_squared={fmt(float(ssb / sst) if sst > 0 else float('nan'))} kruskal_H={fmt(float(hstat))} "
              f"p_H={fmt(float(ph))}", flush=True)


def header(title, purpose):
    print("=" * 100, flush=True)
    print(title, flush=True)
    print(f"Purpose: {purpose}", flush=True)
    print("=" * 100, flush=True)


def run_group(exp, specs, dump_dist=False):
    res = {}
    for label, cfg in specs:
        res[label] = [run_cached(exp, label, cfg, s, dump_dist=dump_dist) for s in SEEDS]
        aggregate(exp, label, res[label])
    return res


SYSTEMS = [("free", dict(system="free")), ("centralized", dict(system="centralized", groups=1)),
           ("hybrid5", dict(system="hybrid", groups=5, split="equal")),
           ("hybrid10", dict(system="hybrid", groups=10, split="equal"))]


def select_device():
    global DEVICE
    c = cfg_of()
    timings = {}
    for dev in (["cpu"] + (["cuda"] if torch.cuda.is_available() else [])):
        DEVICE = torch.device(dev)
        seed_all(0)
        ag = Agent(0, c)
        for _ in range(c["batch"]):
            ag.mem.append(([0.1] * 5, 1, 0.1, [0.1] * 5, False))
        for _ in range(20):
            ag.optimize()
        if dev == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(200):
            ag.optimize()
        if dev == "cuda":
            torch.cuda.synchronize()
        timings[dev] = (time.time() - t0) / 200
    best = min(timings, key=timings.get)
    DEVICE = torch.device(best)
    print("DEVICE_BENCHMARK " + " ".join(f"{k}_ms_per_update={v * 1000:.4f}" for k, v in timings.items())
          + f" selected={best}", flush=True)


def self_test():
    header("PRELIMINARY FUNCTIONAL TEST", "Verify every code path on a reduced configuration before the full run.")
    t0 = time.time()
    small = dict(n_agents=12, days=45, iterations=2, repro_wealth=30)
    variants = [("dqn", {}), ("random", dict(policy="random")), ("rule", dict(policy="rule")),
                ("clip", dict(clip_state=True)), ("clock", dict(eps_clock="continuous", cooldown_clock="continuous")),
                ("noH", dict(happiness_term=False)), ("hp", dict(lr=1e-2, gamma=0.9, buffer=500, batch=16, hidden=32)),
                ("lowfood", dict(food_lo=100, food_hi=300))]
    n = 0
    for sname, sover in SYSTEMS + [("central_equal", dict(system="centralized", groups=1, split="equal")),
                                   ("hybrid5_6040", dict(system="hybrid", groups=5, split="edu6040"))]:
        for vname, vover in variants:
            c = cfg_of(**{**small, **sover, **vover})
            seed_all(7)
            soc = Society(c)
            gday = 0
            for it in range(c["iterations"]):
                itflow = dict(births=0, d_starv=0, d_age=0, loss_sum=0.0, loss_n=0, q_sum=0.0, q_n=0, r_sum=0.0, r_n=0,
                              act=[0, 0, 0], greedy=[0, 0, 0])
                soc.reset_flow()
                for day in range(c["days"]):
                    soc.step_day(day, gday)
                    gday += 1
                f = soc.flow
                for k in ("births", "d_starv", "d_age", "loss_sum", "loss_n", "q_sum", "q_n", "r_sum", "r_n"):
                    itflow[k] += f[k]
                for i in range(3):
                    itflow["act"][i] += f["act"][i]
                    itflow["greedy"][i] += f["greedy"][i]
                soc.total_births += f["births"]
                soc.total_deaths["starvation"] += f["d_starv"]
                soc.total_deaths["age"] += f["d_age"]
                summ, w, h = soc.iteration_summary(itflow)
            assert summ["pop"] == len(soc.agents) == c["n_agents"] + soc.total_births - summ["deaths_cum"], "population accounting"
            assert all(math.isfinite(float(summ[k])) for k in ("pop", "gini", "wealth_mean", "happy_mean", "mortality_ever_lived")), "non-finite metric"
            assert abs(summ["act_collect"] + summ["act_invest"] + summ["act_reproduce"] - 1) < 1e-9 or summ["agent_steps"] == 0
            if c["policy"] == "dqn" and summ["agent_steps"] > 2000:
                assert math.isfinite(summ["loss_mean"]), "loss not finite"
            n += 1
            print(f"TEST_CASE system={sname} variant={vname} pop={summ['pop']} births={summ['births_cum']} "
                  f"deaths={summ['deaths_cum']} gini={summ['gini']:.4f} loss={fmt(summ['loss_mean'], 6)} status=PASS", flush=True)
    c = cfg_of(**small)
    a = simulate("TEST", "determinism", c, 11, [], False)
    b = simulate("TEST", "determinism", c, 11, [], False)
    assert json.dumps(a["summaries"], sort_keys=True) == json.dumps(b["summaries"], sort_keys=True), "non-deterministic run"
    print("TEST_CASE determinism_same_seed status=PASS", flush=True)
    global CACHE_DIR
    keep = CACHE_DIR
    CACHE_DIR = os.path.join(keep, "_selftest")
    r1 = run_cached("TEST", "cache", c, 5)
    r2 = run_cached("TEST", "cache", c, 5)
    assert json.dumps(r1["summaries"], sort_keys=True) == json.dumps(r2["summaries"], sort_keys=True), "cache mismatch"
    for fn in os.listdir(CACHE_DIR):
        os.remove(os.path.join(CACHE_DIR, fn))
    os.rmdir(CACHE_DIR)
    CACHE_DIR = keep
    print("TEST_CASE cache_write_restore status=PASS", flush=True)
    assert abs(gini([1, 1, 1, 1])) < 1e-12 and abs(gini([0, 0, 0, 1]) - 0.75) < 1e-12
    t, p, g, u, pu = compare([1.0, 2.0, 3.0], [4.0, 5.0, 6.5])
    assert math.isfinite(t) and math.isfinite(p) and math.isfinite(g) and math.isfinite(ci95([1.0, 2.0, 3.0]))
    aggregate("TEST", "aggregate", [a, b, a])
    pairwise("TEST", "a", [a, b, a], "b", [r1, r2, r1])
    omnibus("TEST", [("a", [a, b, a]), ("b", [r1, r2, r1])])
    print("TEST_CASE statistics status=PASS", flush=True)
    print(f"PRELIMINARY_TEST_RESULT cases={n + 3} status=ALL_PASSED wall_time_s={time.time() - t0:.1f}", flush=True)


def main():
    header("ENVIRONMENT", "Record software, hardware and global settings.")
    print(f"python={platform.python_version()} torch={torch.__version__} numpy={np.__version__} "
          f"cuda_available={torch.cuda.is_available()} "
          f"gpu={(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')} cpu_count={os.cpu_count()}", flush=True)
    print(f"seeds={SEEDS} cache_dir={CACHE_DIR} code_version={CODE_VERSION}", flush=True)
    print("BASE_CONFIG " + json.dumps(BASE, sort_keys=True), flush=True)
    torch.set_num_threads(1)
    select_device()
    npar = sum(p.numel() for p in QNet(BASE["hidden"]).parameters())
    print(f"MODEL parameters_per_network={npar} networks_per_agent=2 parameters_per_agent={2 * npar}", flush=True)
    print("COMPLEXITY time=O(I*D*N*(B*P)) space=O(N*(P+M*S)) where I=iterations D=days N=population B=batch "
          "P=network_parameters M=replay_capacity S=state_dimension", flush=True)
    self_test()
    t_all = time.time()

    header("EXPERIMENT 1: INDEPENDENT-SEED REPLICATION OF THE FOUR ORGANIZATIONAL SYSTEMS (DQN AGENTS)",
           "Mean, standard deviation and confidence intervals across independently initialized runs; learning diagnostics.")
    e1 = run_group("E1", [(n, cfg_of(**o)) for n, o in SYSTEMS], dump_dist=True)
    omnibus("E1", [(n, e1[n]) for n, _ in SYSTEMS])
    names = [n for n, _ in SYSTEMS]
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            pairwise("E1", names[i], e1[names[i]], names[j], e1[names[j]])

    header("EXPERIMENT 2: NON-LEARNING BASELINES UNDER IDENTICAL CONDITIONS",
           "Uniform random policy and fixed rule-based policy compared against DQN agents in each system.")
    e2 = run_group("E2", [(f"{n}-{p}", cfg_of(policy=p, **o)) for p in ("random", "rule") for n, o in SYSTEMS], dump_dist=True)
    for n in names:
        pairwise("E2", f"{n}-dqn", e1[n], f"{n}-random", e2[f"{n}-random"])
        pairwise("E2", f"{n}-dqn", e1[n], f"{n}-rule", e2[f"{n}-rule"])
        pairwise("E2", f"{n}-random", e2[f"{n}-random"], f"{n}-rule", e2[f"{n}-rule"])

    header("EXPERIMENT 3: SEPARATION OF ORGANIZATIONAL STRUCTURE FROM THE SURPLUS-ALLOCATION RULE",
           "Allocation rule held constant while the number of groups varies, and group structure held constant while the rule varies.")
    spec3 = [(f"groups{g}-{s}", cfg_of(system="hybrid" if g > 1 else "centralized", groups=g, split=s))
             for s in ("equal", "edu6040") for g in (1, 2, 5, 10, 25)
             if not ((g == 1 and s == "edu6040") or (g in (5, 10) and s == "equal"))]
    e3 = run_group("E3", spec3)
    e3["groups1-edu6040"], e3["groups5-equal"], e3["groups10-equal"] = e1["centralized"], e1["hybrid5"], e1["hybrid10"]
    for g in (1, 2, 5, 10, 25):
        pairwise("E3", f"groups{g}-edu6040", e3[f"groups{g}-edu6040"], f"groups{g}-equal", e3[f"groups{g}-equal"])
    for s in ("equal", "edu6040"):
        omnibus("E3", [(f"groups{g}-{s}", e3[f"groups{g}-{s}"]) for g in (1, 2, 5, 10, 25)])

    header("EXPERIMENT 4: MODEL-SPECIFICATION AND REWARD-DESIGN VARIANTS",
           "State clipping, continuous exploration and cooldown calendars, and reward-term ablations in each system.")
    var4 = [("clipstate", dict(clip_state=True)), ("epscontinuous", dict(eps_clock="continuous")),
            ("calendarcontinuous", dict(eps_clock="continuous", cooldown_clock="continuous")),
            ("noreprobonus", dict(r_repro=0.0)), ("nohappinessterm", dict(happiness_term=False))]
    e4 = run_group("E4", [(f"{n}-{vn}", cfg_of(**{**o, **vo})) for vn, vo in var4 for n, o in SYSTEMS])
    for vn, _ in var4:
        for n in names:
            pairwise("E4", f"{n}-baseline", e1[n], f"{n}-{vn}", e4[f"{n}-{vn}"])

    header("EXPERIMENT 5: ONE-AT-A-TIME HYPERPARAMETER SENSITIVITY (FREE AND CENTRALIZED SYSTEMS)",
           "Learning rate, discount factor, exploration decay, replay capacity, batch size and hidden width.")
    var5 = [("lr1e-4", dict(lr=1e-4)), ("lr1e-2", dict(lr=1e-2)), ("gamma0.90", dict(gamma=0.90)),
            ("gamma0.95", dict(gamma=0.95)), ("epsdecay365", dict(eps_decay_days=365)),
            ("epsdecay1460", dict(eps_decay_days=1460)), ("buffer500", dict(buffer=500)), ("buffer5000", dict(buffer=5000)),
            ("batch16", dict(batch=16)), ("batch64", dict(batch=64)), ("hidden32", dict(hidden=32)),
            ("hidden128", dict(hidden=128))]
    two = [x for x in SYSTEMS if x[0] in ("free", "centralized")]
    e5 = run_group("E5", [(f"{n}-{vn}", cfg_of(**{**o, **vo})) for vn, vo in var5 for n, o in two])
    for vn, _ in var5:
        for n, _ in two:
            pairwise("E5", f"{n}-baseline", e1[n], f"{n}-{vn}", e5[f"{n}-{vn}"])

    header("EXPERIMENT 6: INITIAL POPULATION SIZE AND RESOURCE AVAILABILITY",
           "Persistence of the system ranking under different initial population sizes and a reduced food range.")
    var6 = [("n25", dict(n_agents=25)), ("n100", dict(n_agents=100)), ("food500-1500", dict(food_lo=500, food_hi=1500))]
    e6 = run_group("E6", [(f"{n}-{vn}", cfg_of(**{**o, **vo})) for vn, vo in var6 for n, o in SYSTEMS])
    for vn, _ in var6:
        omnibus("E6", [(f"{n}-{vn}", e6[f"{n}-{vn}"]) for n in names])
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                pairwise("E6", f"{names[i]}-{vn}", e6[f"{names[i]}-{vn}"], f"{names[j]}-{vn}", e6[f"{names[j]}-{vn}"])

    header("COMPUTATIONAL COST SUMMARY", "Measured execution time and memory for all experiments.")
    for ename, e in (("E1", e1), ("E2", e2), ("E3", e3), ("E4", e4), ("E5", e5), ("E6", e6)):
        runs = [r for v in e.values() for r in v]
        print(f"{ename} runs={len(runs)} total_wall_time_s={sum(r['wall'] for r in runs):.1f} "
              f"mean_wall_time_s={np.mean([r['wall'] for r in runs]):.1f} total_agent_steps={sum(r['steps'] for r in runs)} "
              f"peak_rss_mb={max(r['rss'] for r in runs):.1f}", flush=True)
    print(f"SESSION_WALL_TIME_S={time.time() - t_all:.1f}", flush=True)
    if torch.cuda.is_available():
        print(f"GPU_PEAK_MEMORY_MB={torch.cuda.max_memory_allocated() / 1e6:.1f}", flush=True)
    print("ALL_EXPERIMENTS_COMPLETED", flush=True)


if __name__ == "__main__":
    main()


ENVIRONMENT
Purpose: Record software, hardware and global settings.
python=3.13.16 torch=2.11.0+cu130 numpy=2.1.3 cuda_available=True gpu=Tesla T4 cpu_count=4
seeds=[42, 123, 456] cache_dir=/kaggle/working/dqn_society_cache code_version=v1
BASE_CONFIG {"batch": 32, "buffer": 1000, "clip_state": false, "collect_eff": 0.08, "cooldown": 365, "cooldown_clock": "iteration", "days": 365, "days_until_death": 2, "edu_gain_collect": 0.01, "edu_gain_pool": 0.005, "edu_hi": 30.0, "edu_lo": 0.0, "eps_clock": "iteration", "eps_decay_days": 730, "eps_end": 0.05, "eps_start": 0.9, "food_hi": 3000, "food_lo": 1000, "gamma": 0.99, "grad_clip": 100.0, "groups": 0, "happiness_term": true, "hidden": 64, "invest_base": 0.5, "invest_gain": 0.1, "iterations": 3, "lr": 0.001, "max_age_hi": 3650, "max_age_lo": 365, "n_agents": 50, "needs_hi": 40, "needs_lo": 10, "partner_prob": 0.15, "policy": "dqn", "pop_cap": 3000, "r_invest_fail": -0.1, "r_invest_success": 0.2, "r_repro": 0.5, "repro_wealth": 200, "split": 